This code is to check and fix train, test and validation subset of bee images. At first, we will generate all the species labels for train, validation, and test subsets. Then we will move the images from validation and test to train if that class is not present in the train subset.

In [1]:
import os
from tqdm import tqdm
import pandas as pd
import shutil

In [2]:
# Class names of every image
fulldata_path = r"/home/c/choton/beemachine/datasets/Bee_Full_Body_Segments/Images"
DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"

# Get list of classes
classes = os.listdir(fulldata_path)
species_dict = {}

for c in classes:
    images_path = os.path.join(fulldata_path, c)
    images = os.listdir(images_path)
    for image in images:
        image_name = image[:-4]
        species_dict[image_name] = c
image_names = list(species_dict.keys())
print(f"Total images for the original data with class labels: {len(image_names)}")

Total images for the original data with class labels: 8029


In [3]:
train_imgs_path = os.path.join(DATA_DIR, 'train', 'images')
val_imgs_path = os.path.join(DATA_DIR, 'valid', 'images')
test_imgs_path = os.path.join(DATA_DIR, 'test', 'images')
train_images = os.listdir(train_imgs_path)
val_images = os.listdir(val_imgs_path)
test_images = os.listdir(test_imgs_path)
print(f"Images in dataset, train: {len(train_images)}, val: {len(val_images)}, test: {len(test_images)}")

Images in dataset, train: 5787, val: 1158, test: 771


In [4]:
# Preprocess image_names ONCE
normalized_species = {
    image_n.replace("._", "-_").replace(".jpg", ""): species_dict[image_n]
    for image_n in image_names
}

Generate labels for train subset

In [5]:
count = 0
species_y = {}
# Loop only once
for image_y in tqdm(train_images):
    for key, species in normalized_species.items():
        if key in image_y:
            count += 1
            species_y[image_y] = species
            break   # IMPORTANT: stop after first match
print(count)

train_dict = {'images': train_images}
train_df = pd.DataFrame(train_dict)
train_df['species'] = train_df['images'].map(species_y)
train_df.head()

  0%|          | 0/5787 [00:00<?, ?it/s]

100%|██████████| 5787/5787 [00:03<00:00, 1453.69it/s]

5787


,images,species
0,Bombus_bifarius_57968807_1_1_jpg.rf.623e797448...,Bombus_bifarius
1,Bombus_polaris_GBIF_iNat_3384911150_7-jpg_jpg....,Bombus_polaris
2,Bombus_pascuorum_iNat_11931463_1-jpg_jpg.rf.e3...,Bombus_pascuorum
3,80JQ703Q20BRKQCRXQCRMQFRRQCQRQDQ603RMQL0IQJRSQ...,Bombus_terricola
4,Bombus_rubicundus_GBIF_iNat_2237442924_3-jpg_j...,Bombus_rubicundus


Generate labels for val subset

In [6]:
count = 0
species_y = {}
# Loop only once
for image_y in tqdm(val_images):
    for key, species in normalized_species.items():
        if key in image_y:
            count += 1
            species_y[image_y] = species
            break   # IMPORTANT: stop after first match
print(count)
valid_dict = {'images': val_images}
valid_df = pd.DataFrame(valid_dict)
valid_df['species'] = valid_df['images'].map(species_y)
valid_df.head()

  0%|          | 0/1158 [00:00<?, ?it/s]

100%|██████████| 1158/1158 [00:00<00:00, 1425.44it/s]

1158


,images,species
0,BBW_Bombus_appositus_23855_e5f978d9-d6e7-4322-...,Bombus_appositus
1,Bombus_lapidarius_iNat_12287097_1-jpg_jpg.rf.3...,Bombus_lapidarius
2,0HGRDZIROZ7RFZXRULHZ9L0R3ZYLNLKROZRZ9LYLNLZZTZ...,Bombus_vosnesenskii
3,Bombus_morio_GBIF_iNat_2603344712_9-jpg_jpg.rf...,Bombus_morio
4,Bombus_pratorum_iNat_21415168_1-jpg_jpg.rf.0db...,Bombus_pratorum


Generate labels for test subset

In [7]:
count = 0
species_y = {}
# Loop only once
for image_y in tqdm(test_images):
    for key, species in normalized_species.items():
        if key in image_y:
            count += 1
            species_y[image_y] = species
            break   # IMPORTANT: stop after first match
print(count)
test_dict = {'images': test_images}
test_df = pd.DataFrame(test_dict)
test_df['species'] = test_df['images'].map(species_y)
test_df.head()

  0%|          | 0/771 [00:00<?, ?it/s]

100%|██████████| 771/771 [00:00<00:00, 1455.62it/s]

771


,images,species
0,Bombus_robustus_GBIF_iNat_2814215056_7-jpg_jpg...,Bombus_robustus
1,5QV06QB0AQ6K8KAK8K1KLKVK7KAKUQLSXK2K5QB0UQ305K...,Bombus_appositus
2,60CQ70CQ6000E0FRQQH0E000IQ3R20FRQQTR20L090YRE0...,Bombus_appositus
3,BBW_Bombus_appositus_28231_846be2d2-b3b3-4947-...,Bombus_appositus
4,Bombus_hortulanus_GBIF_iNat_2028461117_1-jpg_j...,Bombus_hortulanus


In [8]:
n_train = len(set(train_df['species']))
n_val = len(set(valid_df['species']))
n_test = len(set(test_df['species']))
print(f"Total classes in the subsets:")
print("Train:", n_train)
print("Val:", n_val)
print("Test:", n_test)

Total classes in the subsets:
Train: 149
Val: 118
Test: 117


In [9]:
val_train_dif = set(valid_df['species'])-set(train_df['species'])
print("Species that are in valid, but not in train subset:")
val_train_dif

Species that are in valid, but not in train subset:


{'Bombus_laesus',
 'Bombus_lantschouensis',
 'Bombus_modestus',
 'Bombus_quadricolor'}

In [10]:
test_train_dif = set(test_df['species'])-set(train_df['species'])
print("Species that are in test, but not in train subset:")
test_train_dif

Species that are in test, but not in train subset:


{'Bombus_breviceps',
 'Bombus_laesus',
 'Bombus_trinominatus',
 'Bombus_tucumanus',
 'Bombus_tunicatus',
 'Bombus_turkestanicus',
 'Bombus_vestalis',
 'Bombus_wilmattae'}

In [11]:
train_df_path = os.path.join(DATA_DIR, 'train_labels.csv')
val_df_path = os.path.join(DATA_DIR, 'val_labels.csv')
test_df_path = os.path.join(DATA_DIR, 'test_labels.csv')
train_df.to_csv(train_df_path, index=False)
valid_df.to_csv(val_df_path, index=False)
test_df.to_csv(test_df_path, index=False)

In [12]:
import random
random.seed(42)

In [13]:
# ------------------------
# Paths
# ------------------------

DIRS = {
    split: {
        "images": os.path.join(DATA_DIR, split, "images"),
        "labels": os.path.join(DATA_DIR, split, "labels"),
        "masks":  os.path.join(DATA_DIR, split, "masks"),
    }
    for split in ["train", "valid", "test"]
}

CSV_PATHS = {
    "train": os.path.join(DATA_DIR, "train_labels.csv"),
    "valid": os.path.join(DATA_DIR, "val_labels.csv"),
    "test":  os.path.join(DATA_DIR, "test_labels.csv"),
}

# ------------------------
# Load CSVs
# ------------------------
dfs = {k: pd.read_csv(v) for k, v in CSV_PATHS.items()}

# ------------------------
# Helpers
# ------------------------
def move_file_safe(src, dst):
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.move(src, dst)

def move_sample(image_name, src_split, dst_split):
    """
    Move image, YOLO label (.txt), and mask (_m.png) together.
    """
    stem, _ = os.path.splitext(image_name)

    # Image
    move_file_safe(
        os.path.join(DIRS[src_split]["images"], image_name),
        os.path.join(DIRS[dst_split]["images"], image_name),
    )

    # Label
    move_file_safe(
        os.path.join(DIRS[src_split]["labels"], f"{stem}.txt"),
        os.path.join(DIRS[dst_split]["labels"], f"{stem}.txt"),
    )

    # Mask
    move_file_safe(
        os.path.join(DIRS[src_split]["masks"], f"{stem}_m.png"),
        os.path.join(DIRS[dst_split]["masks"], f"{stem}_m.png"),
    )

def safe_train_classes(train_df, min_count=2):
    return (
        train_df["species"]
        .value_counts()
        .loc[lambda x: x >= min_count]
        .index
        .tolist()
    )

# ------------------------
# Main fixing function
# ------------------------
def fix_split(split_name):
    train_df = dfs["train"]
    split_df = dfs[split_name]

    train_classes = set(train_df["species"])
    split_classes = set(split_df["species"])

    missing_classes = split_classes - train_classes
    if not missing_classes:
        print(f"✅ {split_name}: no missing classes")
        return

    print(f"⚠️  {split_name}: fixing classes {missing_classes}")

    # ---- Move missing-class samples INTO train ----
    to_train = split_df[split_df["species"].isin(missing_classes)]

    for _, row in to_train.iterrows():
        move_sample(row.images, split_name, "train")

    dfs["train"] = pd.concat([dfs["train"], to_train], ignore_index=True)
    dfs[split_name] = split_df.drop(to_train.index)

    n_to_replace = len(to_train)

    # ---- Move same number OUT of train ----
    safe_classes = safe_train_classes(dfs["train"])
    candidates = dfs["train"][dfs["train"]["species"].isin(safe_classes)]

    if len(candidates) < n_to_replace:
        raise RuntimeError("❌ Not enough safe samples to rebalance!")

    to_split = candidates.sample(n=n_to_replace, random_state=42)

    for _, row in to_split.iterrows():
        move_sample(row.images, "train", split_name)

    dfs[split_name] = pd.concat([dfs[split_name], to_split], ignore_index=True)
    dfs["train"] = dfs["train"].drop(to_split.index)

    print(f"✅ {split_name}: moved {n_to_replace} samples")

# ------------------------
# Run
# ------------------------
fix_split("valid")
fix_split("test")

# ------------------------
# Save updated CSVs
# ------------------------
for split, df in dfs.items():
    df.to_csv(CSV_PATHS[split], index=False)

print("🎉 Images, labels, and masks successfully rebalanced!")

⚠️  valid: fixing classes {'Bombus_lantschouensis', 'Bombus_modestus', 'Bombus_laesus', 'Bombus_quadricolor'}
✅ valid: moved 4 samples
⚠️  test: fixing classes {'Bombus_wilmattae', 'Bombus_trinominatus', 'Bombus_vestalis', 'Bombus_tucumanus', 'Bombus_turkestanicus', 'Bombus_tunicatus', 'Bombus_breviceps'}
✅ test: moved 60 samples
🎉 Images, labels, and masks successfully rebalanced!
